In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path(r"C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM")

NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "12_product_recommendation_system.ipynb"

DATA_DIR = PROJECT_ROOT / "data"
PREDICTIONS_DIR = DATA_DIR / "predictions"
MODELS_DIR = PROJECT_ROOT / "models"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DB_NAME = "ecommerce_ai_db"
DB_HOST = "localhost"
DB_PORT = 5432

print("Notebook 12: Product Recommendation System")
print(f"Project root: {PROJECT_ROOT}")
print(f"Database: {DB_NAME}")
print(f"Predictions directory: {PREDICTIONS_DIR}")
print(f"Models directory: {MODELS_DIR}")

Notebook 12: Product Recommendation System
Project root: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM
Database: ecommerce_ai_db
Predictions directory: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions
Models directory: C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\models


In [2]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "psycopg2": "psycopg2-binary",
    "sqlalchemy": "sqlalchemy"
}

missing_packages = []

for import_name, package_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        missing_packages.append(package_name)

if missing_packages:
    print("Installing missing packages:")
    print(missing_packages)
    
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            *missing_packages
        ]
    )

print("Required package check completed.")

Required package check completed.


In [3]:
import pandas as pd
import numpy as np

import psycopg2
from psycopg2 import sql
from getpass import getpass

from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

from collections import Counter, defaultdict
from itertools import combinations
from math import sqrt

from sklearn.metrics import precision_score

import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Imports completed successfully.")

Imports completed successfully.


In [4]:
DB_USER = input("Enter PostgreSQL username: ").strip()
DB_PASSWORD = getpass("Enter PostgreSQL password: ")

if not DB_USER:
    raise ValueError("PostgreSQL username cannot be empty.")

if not DB_PASSWORD:
    raise ValueError("PostgreSQL password cannot be empty.")

print("PostgreSQL credentials received securely.")
print("Password has not been printed or stored in the notebook.")

Enter PostgreSQL username:  postgres
Enter PostgreSQL password:  ········


PostgreSQL credentials received securely.
Password has not been printed or stored in the notebook.


In [5]:
def get_psycopg2_connection():
    return psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )


def quote_identifier(identifier):
    return '"' + str(identifier).replace('"', '""') + '"'


def safe_rollback(connection):
    try:
        if connection is not None:
            connection.rollback()
    except Exception:
        pass


def close_connection(connection):
    try:
        if connection is not None:
            connection.close()
    except Exception:
        pass


connection = None

try:
    connection = get_psycopg2_connection()
    
    with connection.cursor() as cursor:
        cursor.execute("SELECT current_database(), current_user;")
        database_name, current_user = cursor.fetchone()
    
    connection.commit()
    
    print("PostgreSQL connection successful.")
    print(f"Connected database: {database_name}")
    print(f"Connected user: {current_user}")

except Exception as error:
    safe_rollback(connection)
    print("PostgreSQL connection failed.")
    raise error

finally:
    close_connection(connection)

PostgreSQL connection successful.
Connected database: ecommerce_ai_db
Connected user: postgres


In [6]:
connection = None

try:
    connection = get_psycopg2_connection()
    
    schemas_query = """
        SELECT schema_name
        FROM information_schema.schemata
        ORDER BY schema_name;
    """
    
    tables_query = """
        SELECT
            table_schema,
            table_name,
            table_type
        FROM information_schema.tables
        WHERE table_type = 'BASE TABLE'
        ORDER BY table_schema, table_name;
    """
    
    columns_query = """
        SELECT
            table_schema,
            table_name,
            ordinal_position,
            column_name,
            data_type,
            udt_name,
            is_nullable
        FROM information_schema.columns
        ORDER BY
            table_schema,
            table_name,
            ordinal_position;
    """
    
    schemas_df = pd.read_sql_query(schemas_query, connection)
    tables_df = pd.read_sql_query(tables_query, connection)
    columns_df = pd.read_sql_query(columns_query, connection)
    
    connection.commit()

except Exception as error:
    safe_rollback(connection)
    raise error

finally:
    close_connection(connection)

print("Discovered schemas:")
display(schemas_df)

print("Discovered tables:")
display(tables_df)

print("Discovered columns:")
display(columns_df.head(100))

Discovered schemas:


,schema_name
0,analytics
1,feature_engineered
2,information_schema
3,pg_catalog
4,pg_toast
5,public
6,sales_analytics


Discovered tables:


,table_schema,table_name,table_type
0,analytics,clv_overall_summary,BASE TABLE
1,analytics,clv_segment_summary,BASE TABLE
2,analytics,clv_validation,BASE TABLE
3,analytics,customer_churn_model_metrics,BASE TABLE
4,analytics,customer_churn_predictions,BASE TABLE
5,analytics,customer_lifetime_value,BASE TABLE
6,analytics,customer_segmentation,BASE TABLE
7,analytics,customer_segmentation_model_evaluation,BASE TABLE
8,analytics,segment_distribution,BASE TABLE
9,analytics,segment_feature_means,BASE TABLE


Discovered columns:


,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable
0,analytics,clv_overall_summary,1,metric,text,text,YES
1,analytics,clv_overall_summary,2,value,text,text,YES
2,analytics,clv_segment_summary,1,clv_segment,text,text,YES
3,analytics,clv_segment_summary,2,customer_count,bigint,int8,YES
4,analytics,clv_segment_summary,3,total_historical_clv,double precision,float8,YES
5,analytics,clv_segment_summary,4,average_historical_clv,double precision,float8,YES
6,analytics,clv_segment_summary,5,median_historical_clv,double precision,float8,YES
7,analytics,clv_segment_summary,6,minimum_historical_clv,double precision,float8,YES
8,analytics,clv_segment_summary,7,maximum_historical_clv,double precision,float8,YES
9,analytics,clv_segment_summary,8,clv_value_share,double precision,float8,YES


In [7]:
def normalize_name(value):
    return (
        str(value)
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def semantic_score(column_name, role):
    name = normalize_name(column_name)
    
    role_tokens = {
        "customer": [
            "customer",
            "buyer",
            "user",
            "client",
            "member",
            "account"
        ],
        "product": [
            "product",
            "item",
            "sku",
            "article",
            "catalog"
        ],
        "order": [
            "order",
            "transaction",
            "purchase",
            "basket",
            "cart"
        ],
        "date": [
            "date",
            "time",
            "timestamp",
            "created",
            "purchase",
            "approved",
            "delivered",
            "shipped"
        ]
    }
    
    tokens = role_tokens[role]
    
    score = 0
    
    for token in tokens:
        if token in name:
            score += 5
    
    if name.endswith("_id") or name == "id":
        score += 2
    
    if role == "date":
        if any(
            data_token in name
            for data_token in ["date", "time", "timestamp"]
        ):
            score += 5
    
    return score


column_candidates = []

for _, row in columns_df.iterrows():
    
    record = {
        "table_schema": row["table_schema"],
        "table_name": row["table_name"],
        "column_name": row["column_name"],
        "data_type": row["data_type"],
        "udt_name": row["udt_name"],
        "is_nullable": row["is_nullable"]
    }
    
    for role in ["customer", "product", "order", "date"]:
        record[f"{role}_score"] = semantic_score(
            row["column_name"],
            role
        )
    
    column_candidates.append(record)

column_candidates_df = pd.DataFrame(column_candidates)

display(
    column_candidates_df.sort_values(
        ["customer_score", "product_score", "order_score"],
        ascending=False
    ).head(100)
)

,table_schema,table_name,column_name,data_type,udt_name,is_nullable,customer_score,product_score,order_score,date_score
23,analytics,customer_churn_predictions,customer_unique_id,text,text,YES,7,2,2,2
32,analytics,customer_lifetime_value,customer_id,text,text,YES,7,2,2,2
38,analytics,customer_segmentation,customer_id,text,text,YES,7,2,2,2
133,feature_engineered,customer_features,customer_id,text,text,YES,7,2,2,2
134,feature_engineered,customer_features,customer_unique_id,text,text,YES,7,2,2,2
170,feature_engineered,order_features,customer_id,text,text,YES,7,2,2,2
2369,public,customers,customer_id,text,text,YES,7,2,2,2
2370,public,customers,customer_unique_id,text,text,YES,7,2,2,2
2399,public,orders,customer_id,text,text,YES,7,2,2,2
809,information_schema,tables,user_defined_type_catalog,name,name,YES,5,5,0,0


In [8]:
connection = None

table_profile_records = []

try:
    connection = get_psycopg2_connection()
    
    for _, table_row in tables_df.iterrows():
        
        schema_name = table_row["table_schema"]
        table_name = table_row["table_name"]
        
        qualified_table = sql.SQL("{}.{}").format(
            sql.Identifier(schema_name),
            sql.Identifier(table_name)
        )
        
        count_query = sql.SQL(
            "SELECT COUNT(*) FROM {}"
        ).format(qualified_table)
        
        with connection.cursor() as cursor:
            cursor.execute(count_query)
            row_count = cursor.fetchone()[0]
        
        table_columns = column_candidates_df[
            (
                column_candidates_df["table_schema"] == schema_name
            )
            &
            (
                column_candidates_df["table_name"] == table_name
            )
        ]
        
        table_profile_records.append({
            "table_schema": schema_name,
            "table_name": table_name,
            "row_count": row_count,
            "customer_columns": int(
                (table_columns["customer_score"] > 0).sum()
            ),
            "product_columns": int(
                (table_columns["product_score"] > 0).sum()
            ),
            "order_columns": int(
                (table_columns["order_score"] > 0).sum()
            ),
            "date_columns": int(
                (table_columns["date_score"] > 0).sum()
            )
        })
    
    connection.commit()

except Exception as error:
    safe_rollback(connection)
    raise error

finally:
    close_connection(connection)

table_profiles_df = pd.DataFrame(table_profile_records)

display(
    table_profiles_df.sort_values(
        "row_count",
        ascending=False
    )
)

,table_schema,table_name,row_count,customer_columns,product_columns,order_columns,date_columns
89,public,geolocation,738332,0,0,0,0
90,public,order_items,112650,4,4,4,5
91,public,order_payments,103886,1,1,1,1
93,public,orders,99441,3,2,12,7
14,feature_engineered,order_features,99441,3,4,20,16
88,public,customers,99441,5,2,2,2
5,analytics,customer_lifetime_value,99441,3,1,1,2
6,analytics,customer_segmentation,99441,4,3,8,5
87,public,customer_churn_predictions,99441,1,0,3,3
12,feature_engineered,customer_features,99441,6,3,10,7


In [9]:
def get_role_candidates(role, minimum_score=5):
    return (
        column_candidates_df[
            column_candidates_df[f"{role}_score"] >= minimum_score
        ]
        .sort_values(
            [f"{role}_score", "table_schema", "table_name"],
            ascending=[False, True, True]
        )
        .reset_index(drop=True)
    )


customer_candidates_df = get_role_candidates("customer")
product_candidates_df = get_role_candidates("product")
order_candidates_df = get_role_candidates("order")
date_candidates_df = get_role_candidates("date")

print("Customer candidates:")
display(customer_candidates_df.head(50))

print("Product candidates:")
display(product_candidates_df.head(50))

print("Order candidates:")
display(order_candidates_df.head(50))

print("Date candidates:")
display(date_candidates_df.head(50))

if customer_candidates_df.empty:
    raise RuntimeError(
        "No validated customer-like identifier candidate was discovered."
    )

if product_candidates_df.empty:
    raise RuntimeError(
        "No validated product-like identifier candidate was discovered."
    )

Customer candidates:


,table_schema,table_name,column_name,data_type,udt_name,is_nullable,customer_score,product_score,order_score,date_score
0,analytics,customer_churn_predictions,customer_unique_id,text,text,YES,7,2,2,2
1,analytics,customer_lifetime_value,customer_id,text,text,YES,7,2,2,2
2,analytics,customer_segmentation,customer_id,text,text,YES,7,2,2,2
3,feature_engineered,customer_features,customer_id,text,text,YES,7,2,2,2
4,feature_engineered,customer_features,customer_unique_id,text,text,YES,7,2,2,2
5,feature_engineered,order_features,customer_id,text,text,YES,7,2,2,2
6,public,customers,customer_id,text,text,YES,7,2,2,2
7,public,customers,customer_unique_id,text,text,YES,7,2,2,2
8,public,orders,customer_id,text,text,YES,7,2,2,2
9,analytics,clv_segment_summary,customer_count,bigint,int8,YES,5,0,0,0


Product candidates:


,table_schema,table_name,column_name,data_type,udt_name,is_nullable,customer_score,product_score,order_score,date_score
0,feature_engineered,product_features,product_id,text,text,YES,2,7,2,2
1,public,order_items,order_item_id,bigint,int8,YES,2,7,7,2
2,public,order_items,product_id,text,text,YES,2,7,2,2
3,public,products,product_id,text,text,YES,2,7,2,2
4,analytics,customer_segmentation,total_items_purchased,double precision,float8,YES,0,5,5,5
5,analytics,segment_feature_means,total_items_purchased,double precision,float8,YES,0,5,5,5
6,analytics,segment_summary,mean_total_items_purchased,double precision,float8,YES,0,5,5,5
7,analytics,segment_summary,median_total_items_purchased,double precision,float8,YES,0,5,5,5
8,analytics,segment_summary,min_total_items_purchased,double precision,float8,YES,0,5,5,5
9,analytics,segment_summary,max_total_items_purchased,double precision,float8,YES,0,5,5,5


Order candidates:


,table_schema,table_name,column_name,data_type,udt_name,is_nullable,customer_score,product_score,order_score,date_score
0,feature_engineered,order_features,order_purchase_timestamp,timestamp without time zone,timestamp,YES,0,0,10,20
1,public,orders,order_purchase_timestamp,text,text,YES,0,0,10,20
2,feature_engineered,order_features,order_id,text,text,YES,2,2,7,2
3,public,order_items,order_id,text,text,YES,2,2,7,2
4,public,order_items,order_item_id,bigint,int8,YES,2,7,7,2
5,public,order_payments,order_id,text,text,YES,2,2,7,2
6,public,order_reviews,order_id,text,text,YES,2,2,7,2
7,public,orders,order_id,text,text,YES,2,2,7,2
8,analytics,customer_segmentation,total_orders,double precision,float8,YES,0,0,5,0
9,analytics,customer_segmentation,average_order_value,double precision,float8,YES,0,0,5,0


Date candidates:


,table_schema,table_name,column_name,data_type,udt_name,is_nullable,customer_score,product_score,order_score,date_score
0,feature_engineered,order_features,order_purchase_timestamp,timestamp without time zone,timestamp,YES,0,0,10,20
1,public,orders,order_purchase_timestamp,text,text,YES,0,0,10,20
2,feature_engineered,customer_features,first_purchase_date,timestamp without time zone,timestamp,YES,0,0,5,15
3,feature_engineered,customer_features,last_purchase_date,timestamp without time zone,timestamp,YES,0,0,5,15
4,feature_engineered,order_features,order_delivered_carrier_date,timestamp without time zone,timestamp,YES,0,0,5,15
5,feature_engineered,order_features,order_delivered_customer_date,timestamp without time zone,timestamp,YES,5,0,5,15
6,feature_engineered,order_features,purchase_date,date,date,YES,0,0,5,15
7,information_schema,attributes,datetime_precision,integer,int4,YES,0,0,0,15
8,information_schema,columns,datetime_precision,integer,int4,YES,0,0,0,15
9,information_schema,domains,datetime_precision,integer,int4,YES,0,0,0,15


In [10]:
def choose_best_column(
    schema_name,
    table_name,
    role,
    preferred_data_types=None
):
    
    candidates = column_candidates_df[
        (
            column_candidates_df["table_schema"] == schema_name
        )
        &
        (
            column_candidates_df["table_name"] == table_name
        )
        &
        (
            column_candidates_df[f"{role}_score"] > 0
        )
    ].copy()
    
    if candidates.empty:
        return None
    
    if preferred_data_types:
        preferred = candidates[
            candidates["data_type"].isin(preferred_data_types)
        ]
        
        if not preferred.empty:
            candidates = preferred
    
    candidates = candidates.sort_values(
        f"{role}_score",
        ascending=False
    )
    
    return candidates.iloc[0]["column_name"]


def get_table_columns(schema_name, table_name):
    return columns_df[
        (
            columns_df["table_schema"] == schema_name
        )
        &
        (
            columns_df["table_name"] == table_name
        )
    ].copy()


def find_common_key(
    left_schema,
    left_table,
    right_schema,
    right_table,
    role="order"
):
    
    left_columns = get_table_columns(
        left_schema,
        left_table
    )
    
    right_columns = get_table_columns(
        right_schema,
        right_table
    )
    
    common_keys = []
    
    for _, left_row in left_columns.iterrows():
        
        for _, right_row in right_columns.iterrows():
            
            same_name = (
                normalize_name(left_row["column_name"])
                ==
                normalize_name(right_row["column_name"])
            )
            
            same_type = (
                left_row["data_type"] == right_row["data_type"]
            ) or (
                left_row["udt_name"] == right_row["udt_name"]
            )
            
            semantic_match = (
                semantic_score(
                    left_row["column_name"],
                    role
                ) > 0
                or
                semantic_score(
                    right_row["column_name"],
                    role
                ) > 0
            )
            
            if same_name and same_type and semantic_match:
                common_keys.append({
                    "left_column": left_row["column_name"],
                    "right_column": right_row["column_name"],
                    "data_type": left_row["data_type"]
                })
    
    if not common_keys:
        return None
    
    return common_keys[0]

In [11]:
# Discover a direct customer-product interaction table first.

direct_interaction_candidates = []

for _, table_row in tables_df.iterrows():
    
    schema_name = table_row["table_schema"]
    table_name = table_row["table_name"]
    
    table_columns = get_table_columns(
        schema_name,
        table_name
    )
    
    customer_columns = table_columns[
        table_columns["column_name"].apply(
            lambda x: semantic_score(x, "customer") > 0
        )
    ]
    
    product_columns = table_columns[
        table_columns["column_name"].apply(
            lambda x: semantic_score(x, "product") > 0
        )
    ]
    
    if (
        not customer_columns.empty
        and
        not product_columns.empty
    ):
        
        best_customer = customer_columns.sort_values(
            by="column_name"
        ).iloc[0]
        
        best_product = product_columns.sort_values(
            by="column_name"
        ).iloc[0]
        
        direct_interaction_candidates.append({
            "table_schema": schema_name,
            "table_name": table_name,
            "customer_column": best_customer["column_name"],
            "customer_data_type": best_customer["data_type"],
            "product_column": best_product["column_name"],
            "product_data_type": best_product["data_type"],
            "row_count": table_profiles_df[
                (
                    table_profiles_df["table_schema"]
                    == schema_name
                )
                &
                (
                    table_profiles_df["table_name"]
                    == table_name
                )
            ]["row_count"].iloc[0]
        })

direct_interaction_candidates_df = pd.DataFrame(
    direct_interaction_candidates
)

display(direct_interaction_candidates_df)

,table_schema,table_name,customer_column,customer_data_type,product_column,product_data_type,row_count
0,analytics,customer_churn_predictions,customer_unique_id,text,customer_unique_id,text,86924
1,analytics,customer_lifetime_value,customer_id,text,customer_id,text,99441
2,analytics,customer_segmentation,cluster_id,bigint,cluster_id,bigint,99441
3,analytics,segment_distribution,cluster_id,bigint,cluster_id,bigint,2
4,analytics,segment_feature_means,cluster_id,bigint,cluster_id,bigint,2
5,analytics,segment_summary,cluster_id,bigint,cluster_id,bigint,2
6,feature_engineered,customer_features,customer_city,text,customer_id,text,99441
7,feature_engineered,order_features,customer_id,text,customer_id,text,99441
8,feature_engineered,product_features,product_id,text,average_product_review_score,double precision,32951
9,feature_engineered,seller_features,seller_id,text,average_item_price,double precision,3095


In [12]:
interaction_source = None

if not direct_interaction_candidates_df.empty:
    
    direct_interaction_candidates_df = (
        direct_interaction_candidates_df
        .sort_values(
            "row_count",
            ascending=False
        )
        .reset_index(drop=True)
    )
    
    selected = direct_interaction_candidates_df.iloc[0]
    
    interaction_source = {
        "mode": "direct",
        "interaction_schema": selected["table_schema"],
        "interaction_table": selected["table_name"],
        "customer_column": selected["customer_column"],
        "product_column": selected["product_column"],
        "order_schema": None,
        "order_table": None,
        "order_column": None,
        "line_schema": None,
        "line_table": None,
        "line_order_column": None,
        "line_product_column": None
    }

else:
    
    if order_candidates_df.empty:
        raise RuntimeError(
            "No order or transaction identifier was discovered, "
            "and no direct customer-product interaction table exists."
        )
    
    order_table_options = []
    
    for _, candidate in order_candidates_df.iterrows():
        
        schema_name = candidate["table_schema"]
        table_name = candidate["table_name"]
        
        table_columns = get_table_columns(
            schema_name,
            table_name
        )
        
        customer_columns = table_columns[
            table_columns["column_name"].apply(
                lambda x: semantic_score(x, "customer") > 0
            )
        ]
        
        order_columns = table_columns[
            table_columns["column_name"].apply(
                lambda x: semantic_score(x, "order") > 0
            )
        ]
        
        if (
            not customer_columns.empty
            and
            not order_columns.empty
        ):
            
            best_customer = customer_columns.sort_values(
                by="column_name"
            ).iloc[0]
            
            best_order = order_columns.sort_values(
                by="column_name"
            ).iloc[0]
            
            order_table_options.append({
                "schema": schema_name,
                "table": table_name,
                "customer_column": best_customer["column_name"],
                "order_column": best_order["column_name"]
            })
    
    if not order_table_options:
        raise RuntimeError(
            "No validated order table containing both a customer-like "
            "identifier and an order-like identifier was discovered."
        )
    
    selected_order = order_table_options[0]
    
    line_table_options = []
    
    for _, candidate in product_candidates_df.iterrows():
        
        schema_name = candidate["table_schema"]
        table_name = candidate["table_name"]
        
        table_columns = get_table_columns(
            schema_name,
            table_name
        )
        
        product_columns = table_columns[
            table_columns["column_name"].apply(
                lambda x: semantic_score(x, "product") > 0
            )
        ]
        
        if product_columns.empty:
            continue
        
        common_order_key = find_common_key(
            selected_order["schema"],
            selected_order["table"],
            schema_name,
            table_name,
            role="order"
        )
        
        if common_order_key:
            
            best_product = product_columns.sort_values(
                by="column_name"
            ).iloc[0]
            
            line_table_options.append({
                "schema": schema_name,
                "table": table_name,
                "order_column": common_order_key["right_column"],
                "product_column": best_product["column_name"]
            })
    
    if not line_table_options:
        raise RuntimeError(
            "A customer-order table was discovered, but no validated "
            "order-product relationship could be established."
        )
    
    selected_line = line_table_options[0]
    
    interaction_source = {
        "mode": "relational",
        "interaction_schema": None,
        "interaction_table": None,
        "customer_column": None,
        "product_column": None,
        "order_schema": selected_order["schema"],
        "order_table": selected_order["table"],
        "order_column": selected_order["order_column"],
        "line_schema": selected_line["schema"],
        "line_table": selected_line["table"],
        "line_order_column": selected_line["order_column"],
        "line_product_column": selected_line["product_column"]
    }

print("Validated interaction source:")
display(pd.DataFrame([interaction_source]))

Validated interaction source:


,mode,interaction_schema,interaction_table,customer_column,product_column,order_schema,order_table,order_column,line_schema,line_table,line_order_column,line_product_column
0,direct,public,order_items,order_id,order_id,None,None,None,None,None,None,None


In [13]:
connection = None

try:
    connection = get_psycopg2_connection()
    
    if interaction_source["mode"] == "direct":
        
        source_schema = interaction_source["interaction_schema"]
        source_table = interaction_source["interaction_table"]
        customer_column = interaction_source["customer_column"]
        product_column = interaction_source["product_column"]
        
        source_columns = get_table_columns(
            source_schema,
            source_table
        )
        
        possible_date_columns = source_columns[
            source_columns["column_name"].apply(
                lambda x: semantic_score(x, "date") > 0
            )
        ]
        
        date_column = None
        
        if not possible_date_columns.empty:
            date_column = (
                possible_date_columns
                .sort_values("ordinal_position")
                .iloc[0]["column_name"]
            )
        
        select_parts = [
            sql.SQL("{} AS customer_id").format(
                sql.Identifier(customer_column)
            ),
            sql.SQL("{} AS product_id").format(
                sql.Identifier(product_column)
            )
        ]
        
        if date_column:
            select_parts.append(
                sql.SQL("{} AS interaction_date").format(
                    sql.Identifier(date_column)
                )
            )
        else:
            select_parts.append(
                sql.SQL("NULL AS interaction_date")
            )
        
        query = sql.SQL(
            "SELECT {} FROM {}.{}"
        ).format(
            sql.SQL(", ").join(select_parts),
            sql.Identifier(source_schema),
            sql.Identifier(source_table)
        )
        
        with connection.cursor() as cursor:
            cursor.execute(query)
            rows = cursor.fetchall()
            column_names = [
                description[0]
                for description in cursor.description
            ]
        
        interactions_df = pd.DataFrame(
            rows,
            columns=column_names
        )
        
        interaction_date_source = date_column
    
    else:
        
        order_schema = interaction_source["order_schema"]
        order_table = interaction_source["order_table"]
        order_customer_column = interaction_source["customer_column"]
        order_column = interaction_source["order_column"]
        
        line_schema = interaction_source["line_schema"]
        line_table = interaction_source["line_table"]
        line_order_column = interaction_source["line_order_column"]
        line_product_column = interaction_source["line_product_column"]
        
        order_columns = get_table_columns(
            order_schema,
            order_table
        )
        
        possible_date_columns = order_columns[
            order_columns["column_name"].apply(
                lambda x: semantic_score(x, "date") > 0
            )
        ]
        
        date_column = None
        
        if not possible_date_columns.empty:
            date_column = (
                possible_date_columns
                .sort_values("ordinal_position")
                .iloc[0]["column_name"]
            )
        
        if date_column:
            date_expression = sql.SQL(
                "o.{} AS interaction_date"
            ).format(
                sql.Identifier(date_column)
            )
        else:
            date_expression = sql.SQL(
                "NULL AS interaction_date"
            )
        
        query = sql.SQL(
            """
            SELECT
                o.{customer_column} AS customer_id,
                l.{product_column} AS product_id,
                {date_expression}
            FROM {order_schema}.{order_table} AS o
            INNER JOIN {line_schema}.{line_table} AS l
                ON o.{order_column} = l.{line_order_column}
            """
        ).format(
            customer_column=sql.Identifier(
                order_customer_column
            ),
            product_column=sql.Identifier(
                line_product_column
            ),
            date_expression=date_expression,
            order_schema=sql.Identifier(
                order_schema
            ),
            order_table=sql.Identifier(
                order_table
            ),
            line_schema=sql.Identifier(
                line_schema
            ),
            line_table=sql.Identifier(
                line_table
            ),
            order_column=sql.Identifier(
                order_column
            ),
            line_order_column=sql.Identifier(
                line_order_column
            )
        )
        
        with connection.cursor() as cursor:
            cursor.execute(query)
            rows = cursor.fetchall()
            column_names = [
                description[0]
                for description in cursor.description
            ]
        
        interactions_df = pd.DataFrame(
            rows,
            columns=column_names
        )
        
        interaction_date_source = date_column
    
    connection.commit()

except Exception as error:
    safe_rollback(connection)
    raise error

finally:
    close_connection(connection)

print(f"Raw interaction rows retrieved: {len(interactions_df):,}")
print(f"Date column used: {interaction_date_source}")
display(interactions_df.head())

Raw interaction rows retrieved: 112,650
Date column used: order_id


,customer_id,product_id,interaction_date
0,00010242fe8c5a6d1ba2dd792cb16214,00010242fe8c5a6d1ba2dd792cb16214,00010242fe8c5a6d1ba2dd792cb16214
1,00018f77f2f0320c557190d7a144bdd3,00018f77f2f0320c557190d7a144bdd3,00018f77f2f0320c557190d7a144bdd3
2,000229ec398224ef6ca0657da4fc703e,000229ec398224ef6ca0657da4fc703e,000229ec398224ef6ca0657da4fc703e
3,00024acbcdf0a6daa1e931b038114c75,00024acbcdf0a6daa1e931b038114c75,00024acbcdf0a6daa1e931b038114c75
4,00042b26cf59d7ce69dfabb4e55b4fd9,00042b26cf59d7ce69dfabb4e55b4fd9,00042b26cf59d7ce69dfabb4e55b4fd9


In [14]:
interaction_validation = {}

interaction_validation["raw_rows"] = len(interactions_df)

interaction_validation["missing_customer_ids"] = int(
    interactions_df["customer_id"].isna().sum()
)

interaction_validation["missing_product_ids"] = int(
    interactions_df["product_id"].isna().sum()
)

interaction_validation["duplicate_rows"] = int(
    interactions_df.duplicated(
        subset=["customer_id", "product_id"]
    ).sum()
)

interaction_validation["unique_customers_before_cleaning"] = (
    interactions_df["customer_id"]
    .dropna()
    .nunique()
)

interaction_validation["unique_products_before_cleaning"] = (
    interactions_df["product_id"]
    .dropna()
    .nunique()
)

if interactions_df["interaction_date"].notna().any():
    
    interactions_df["interaction_date"] = pd.to_datetime(
        interactions_df["interaction_date"],
        errors="coerce"
    )
    
    interaction_validation["date_min"] = (
        interactions_df["interaction_date"].min()
    )
    
    interaction_validation["date_max"] = (
        interactions_df["interaction_date"].max()
    )
    
else:
    
    interaction_validation["date_min"] = None
    interaction_validation["date_max"] = None

print("Interaction data-quality validation:")
display(pd.DataFrame([interaction_validation]))

if interaction_validation["raw_rows"] == 0:
    raise RuntimeError(
        "No customer-product interactions were retrieved."
    )

if interaction_validation["missing_customer_ids"] == (
    interaction_validation["raw_rows"]
):
    raise RuntimeError(
        "All customer identifiers are missing."
    )

if interaction_validation["missing_product_ids"] == (
    interaction_validation["raw_rows"]
):
    raise RuntimeError(
        "All product identifiers are missing."
    )

Interaction data-quality validation:


,raw_rows,missing_customer_ids,missing_product_ids,duplicate_rows,unique_customers_before_cleaning,unique_products_before_cleaning,date_min,date_max
0,112650,0,0,13984,98666,98666,NaT,NaT


In [15]:
interactions_df = interactions_df.dropna(
    subset=["customer_id", "product_id"]
).copy()

interactions_df["customer_id"] = (
    interactions_df["customer_id"]
    .astype(str)
    .str.strip()
)

interactions_df["product_id"] = (
    interactions_df["product_id"]
    .astype(str)
    .str.strip()
)

interactions_df = interactions_df[
    (
        interactions_df["customer_id"] != ""
    )
    &
    (
        interactions_df["product_id"] != ""
    )
].copy()

interactions_df = (
    interactions_df
    .sort_values(
        by=["customer_id", "product_id", "interaction_date"],
        na_position="first"
    )
    .drop_duplicates(
        subset=["customer_id", "product_id"],
        keep="last"
    )
    .reset_index(drop=True)
)

interactions_df["interaction_value"] = 1

print("Clean customer-product interactions:")
print(f"Rows: {len(interactions_df):,}")
print(
    f"Customers: {interactions_df['customer_id'].nunique():,}"
)
print(
    f"Products: {interactions_df['product_id'].nunique():,}"
)

display(interactions_df.head())

Clean customer-product interactions:
Rows: 98,666
Customers: 98,666
Products: 98,666


,customer_id,product_id,interaction_date,interaction_value
0,00010242fe8c5a6d1ba2dd792cb16214,00010242fe8c5a6d1ba2dd792cb16214,NaT,1
1,00018f77f2f0320c557190d7a144bdd3,00018f77f2f0320c557190d7a144bdd3,NaT,1
2,000229ec398224ef6ca0657da4fc703e,000229ec398224ef6ca0657da4fc703e,NaT,1
3,00024acbcdf0a6daa1e931b038114c75,00024acbcdf0a6daa1e931b038114c75,NaT,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,00042b26cf59d7ce69dfabb4e55b4fd9,NaT,1


In [16]:
customer_interaction_counts = (
    interactions_df
    .groupby("customer_id")
    .agg(
        unique_products=("product_id", "nunique"),
        interaction_count=("interaction_value", "sum")
    )
    .reset_index()
)

product_interaction_counts = (
    interactions_df
    .groupby("product_id")
    .agg(
        unique_customers=("customer_id", "nunique"),
        interaction_count=("interaction_value", "sum")
    )
    .reset_index()
)

number_of_customers = (
    interactions_df["customer_id"].nunique()
)

number_of_products = (
    interactions_df["product_id"].nunique()
)

number_of_interactions = len(interactions_df)

possible_interactions = (
    number_of_customers
    *
    number_of_products
)

sparsity = 1 - (
    number_of_interactions
    /
    possible_interactions
)

interaction_quality_summary = pd.DataFrame([{
    "customers": number_of_customers,
    "products": number_of_products,
    "interactions": number_of_interactions,
    "mean_products_per_customer": (
        customer_interaction_counts["unique_products"].mean()
    ),
    "median_products_per_customer": (
        customer_interaction_counts["unique_products"].median()
    ),
    "mean_customers_per_product": (
        product_interaction_counts["unique_customers"].mean()
    ),
    "median_customers_per_product": (
        product_interaction_counts["unique_customers"].median()
    ),
    "sparsity": sparsity,
    "density": 1 - sparsity
}])

print("Interaction structure summary:")
display(interaction_quality_summary)

print("Customer interaction distribution:")
display(
    customer_interaction_counts[
        "unique_products"
    ].describe()
)

print("Product interaction distribution:")
display(
    product_interaction_counts[
        "unique_customers"
    ].describe()
)

Interaction structure summary:


,customers,products,interactions,mean_products_per_customer,median_products_per_customer,mean_customers_per_product,median_customers_per_product,sparsity,density
0,98666,98666,98666,1.0,1.0,1.0,1.0,0.99999,0.00001


Customer interaction distribution:


count    98666.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: unique_products, dtype: float64

Product interaction distribution:


count    98666.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: unique_customers, dtype: float64

In [17]:
TOP_K = 10

MIN_HISTORY_FOR_PERSONALIZATION = 2

MIN_CUSTOMERS_FOR_PERSONALIZATION = 100

customer_history_counts = (
    interactions_df
    .groupby("customer_id")["product_id"]
    .nunique()
)

customers_with_multiple_products = int(
    (
        customer_history_counts
        >= MIN_HISTORY_FOR_PERSONALIZATION
    ).sum()
)

cooccurrence_pairs = 0

for customer_id, group in (
    interactions_df
    .groupby("customer_id")
):
    
    products = list(
        group["product_id"].unique()
    )
    
    if len(products) >= 2:
        cooccurrence_pairs += (
            len(list(combinations(products, 2)))
        )

personalization_supported = (
    customers_with_multiple_products
    >= MIN_CUSTOMERS_FOR_PERSONALIZATION
    and
    cooccurrence_pairs > 0
)

method_decision = pd.DataFrame([{
    "customers_with_multiple_products": (
        customers_with_multiple_products
    ),
    "cooccurrence_pairs": cooccurrence_pairs,
    "personalization_supported": (
        personalization_supported
    ),
    "selected_personalized_method": (
        "item-based co-occurrence similarity"
        if personalization_supported
        else "not supported"
    ),
    "fallback_method": "popular-product recommendations"
}])

display(method_decision)

if not personalization_supported:
    print(
        "The data does not safely support a meaningful "
        "personalized method under the validation rules."
    )
    print(
        "The recommendation system will use the popularity "
        "baseline with validated cold-start fallback logic."
    )
else:
    print(
        "The data supports a personalized item-based "
        "co-occurrence recommendation method."
    )

,customers_with_multiple_products,cooccurrence_pairs,personalization_supported,selected_personalized_method,fallback_method
0,0,0,False,not supported,popular-product recommendations


The data does not safely support a meaningful personalized method under the validation rules.
The recommendation system will use the popularity baseline with validated cold-start fallback logic.


In [18]:
def build_popularity_ranking(training_interactions):
    
    popularity = (
        training_interactions
        .groupby("product_id")
        .agg(
            interaction_count=(
                "interaction_value",
                "sum"
            ),
            unique_customers=(
                "customer_id",
                "nunique"
            )
        )
        .reset_index()
    )
    
    popularity["popularity_score"] = (
        popularity["interaction_count"]
        /
        popularity["interaction_count"].max()
    )
    
    popularity = popularity.sort_values(
        [
            "interaction_count",
            "unique_customers",
            "product_id"
        ],
        ascending=[False, False, True]
    ).reset_index(drop=True)
    
    popularity["rank"] = (
        np.arange(len(popularity))
        + 1
    )
    
    return popularity


popularity_ranking = build_popularity_ranking(
    interactions_df
)

print("Popularity baseline created.")

display(
    popularity_ranking.head(TOP_K)
)

Popularity baseline created.


,product_id,interaction_count,unique_customers,popularity_score,rank
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1.0,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,1.0,2
2,000229ec398224ef6ca0657da4fc703e,1,1,1.0,3
3,00024acbcdf0a6daa1e931b038114c75,1,1,1.0,4
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1.0,5
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,1,1.0,6
6,00054e8431b9d7675808bcb819fb4a32,1,1,1.0,7
7,000576fe39319847cbb9d288c5617fa6,1,1,1.0,8
8,0005a1a1728c9d785b8e2b08b904576c,1,1,1.0,9
9,0005f50442cb953dcd1d21e1fb923495,1,1,1.0,10


In [26]:
# Safe temporal train/test split

def temporal_train_test_split(interactions):

    data = interactions.copy()

    required_columns = {
        "customer_id",
        "product_id",
        "interaction_date"
    }

    missing_columns = (
        required_columns
        - set(data.columns)
    )

    if missing_columns:

        return (
            None,
            None,
            "missing_required_columns"
        )

    if data.empty:

        return (
            None,
            None,
            "empty_interactions"
        )

    if data["interaction_date"].isna().all():

        return (
            None,
            None,
            "no_valid_date"
        )

    data = (
        data
        .dropna(
            subset=[
                "customer_id",
                "product_id",
                "interaction_date"
            ]
        )
        .copy()
    )

    if data.empty:

        return (
            None,
            None,
            "no_valid_interactions"
        )

    data = (
        data
        .sort_values(
            [
                "customer_id",
                "interaction_date"
            ]
        )
    )

    # Keep the latest interaction for each
    # customer-product pair.
    customer_product_latest = (

        data
        .sort_values(
            "interaction_date"
        )
        .drop_duplicates(
            subset=[
                "customer_id",
                "product_id"
            ],
            keep="last"
        )
    )

    customer_product_counts = (

        customer_product_latest
        .groupby(
            "customer_id"
        )["product_id"]
        .nunique()
    )

    eligible_customers = (

        customer_product_counts[
            customer_product_counts >= 2
        ]
        .index
    )

    if len(eligible_customers) == 0:

        return (
            None,
            None,
            "insufficient_customer_history"
        )

    eligible_data = (

        customer_product_latest[
            customer_product_latest[
                "customer_id"
            ].isin(
                eligible_customers
            )
        ]
        .copy()
    )

    test_indices = (

        eligible_data
        .sort_values(
            [
                "customer_id",
                "interaction_date"
            ]
        )
        .groupby(
            "customer_id"
        )
        .tail(1)
        .index
    )

    test_data = (

        eligible_data
        .loc[
            test_indices
        ]
        .copy()
    )

    train_data = (

        eligible_data
        .drop(
            test_indices
        )
        .copy()
    )

    if train_data.empty:

        return (
            None,
            None,
            "empty_training_set"
        )

    if test_data.empty:

        return (
            None,
            None,
            "empty_test_set"
        )

    return (
        train_data,
        test_data,
        "temporal"
    )


(
    train_interactions,
    test_interactions,
    split_method
) = temporal_train_test_split(
    interactions_df
)


if split_method == "temporal":

    evaluation_supported = True

    print(
        "✅ Valid temporal evaluation split created."
    )

else:

    evaluation_supported = False

    train_interactions = pd.DataFrame(
        columns=interactions_df.columns
    )

    test_interactions = pd.DataFrame(
        columns=interactions_df.columns
    )

    print(
        "⚠️ Personalized evaluation is not supported "
        "by the validated interaction structure."
    )

    print(
        f"Reason: {split_method}"
    )

    print(
        "The notebook will not fabricate "
        "Precision@K, Recall@K, or Hit Rate@K."
    )


print(
    f"Evaluation split method: {split_method}"
)

print(
    f"Training interactions: "
    f"{len(train_interactions):,}"
)

print(
    f"Testing interactions: "
    f"{len(test_interactions):,}"
)

print(
    f"Training customers: "
    f"{train_interactions['customer_id'].nunique():,}"
)

print(
    f"Testing customers: "
    f"{test_interactions['customer_id'].nunique():,}"
)

⚠️ Personalized evaluation is not supported by the validated interaction structure.
Reason: no_valid_date
The notebook will not fabricate Precision@K, Recall@K, or Hit Rate@K.
Evaluation split method: no_valid_date
Training interactions: 0
Testing interactions: 0
Training customers: 0
Testing customers: 0


In [27]:
# Build popularity baseline

if interactions_df.empty:

    raise RuntimeError(
        "Cannot build popularity recommendations. "
        "The validated interaction dataset is empty."
    )


popularity_source_interactions = (
    interactions_df.copy()
)


training_popularity = (
    build_popularity_ranking(
        popularity_source_interactions
    )
)


if training_popularity.empty:

    raise RuntimeError(
        "Popularity ranking is empty even though "
        "validated interactions exist."
    )


def popularity_recommendations(
    popularity_df,
    seen_products=None,
    k=10
):

    if seen_products is None:

        seen_products = set()

    recommendations = []

    for _, row in popularity_df.iterrows():

        product_id = row["product_id"]

        if product_id in seen_products:

            continue

        recommendations.append(
            {
                "product_id": product_id,
                "score": float(
                    row["popularity_score"]
                ),
                "method": (
                    "popularity_baseline"
                )
            }
        )

        if len(
            recommendations
        ) >= k:

            break

    return recommendations


global_popular_recommendations = (
    popularity_recommendations(
        training_popularity,
        seen_products=set(),
        k=TOP_K
    )
)


if not global_popular_recommendations:

    raise RuntimeError(
        "No popularity recommendations "
        "were generated."
    )


print(
    "✅ Popularity baseline successfully generated."
)

display(
    pd.DataFrame(
        global_popular_recommendations
    )
)

✅ Popularity baseline successfully generated.


,product_id,score,method
0,00010242fe8c5a6d1ba2dd792cb16214,1.0,popularity_baseline
1,00018f77f2f0320c557190d7a144bdd3,1.0,popularity_baseline
2,000229ec398224ef6ca0657da4fc703e,1.0,popularity_baseline
3,00024acbcdf0a6daa1e931b038114c75,1.0,popularity_baseline
4,00042b26cf59d7ce69dfabb4e55b4fd9,1.0,popularity_baseline
5,00048cc3ae777c65dbb7d2a0634bc1ea,1.0,popularity_baseline
6,00054e8431b9d7675808bcb819fb4a32,1.0,popularity_baseline
7,000576fe39319847cbb9d288c5617fa6,1.0,popularity_baseline
8,0005a1a1728c9d785b8e2b08b904576c,1.0,popularity_baseline
9,0005f50442cb953dcd1d21e1fb923495,1.0,popularity_baseline


In [28]:
def build_item_cooccurrence_model(
    training_interactions,
    max_products_per_customer=100
):
    
    customer_product_groups = (
        training_interactions
        .groupby("customer_id")
    )
    
    product_customer_count = (
        training_interactions
        .groupby("product_id")["customer_id"]
        .nunique()
        .to_dict()
    )
    
    cooccurrence = defaultdict(Counter)
    
    for customer_id, group in customer_product_groups:
        
        products = list(
            group["product_id"].unique()
        )
        
        if len(products) > max_products_per_customer:
            
            products = sorted(
                products,
                key=lambda product_id: (
                    product_customer_count.get(
                        product_id,
                        0
                    )
                ),
                reverse=True
            )[
                :max_products_per_customer
            ]
        
        if len(products) < 2:
            continue
        
        for product_a, product_b in combinations(
            sorted(products),
            2
        ):
            
            cooccurrence[
                product_a
            ][product_b] += 1
            
            cooccurrence[
                product_b
            ][product_a] += 1
    
    product_frequency = (
        training_interactions
        .groupby("product_id")["customer_id"]
        .nunique()
        .to_dict()
    )
    
    similarity = defaultdict(dict)
    
    for product_a, neighbors in cooccurrence.items():
        
        for product_b, co_count in neighbors.items():
            
            denominator = sqrt(
                product_frequency.get(
                    product_a,
                    0
                )
                *
                product_frequency.get(
                    product_b,
                    0
                )
            )
            
            if denominator > 0:
                
                similarity[
                    product_a
                ][product_b] = (
                    co_count
                    /
                    denominator
                )
    
    return similarity, product_frequency


if personalization_supported:
    
    item_similarity, training_product_frequency = (
        build_item_cooccurrence_model(
            train_interactions
        )
    )
    
    print(
        "Item-based co-occurrence model created."
    )
    
    print(
        f"Products with similarity relationships: "
        f"{len(item_similarity):,}"
    )

else:
    
    item_similarity = {}
    training_product_frequency = {}
    
    print(
        "Personalized model was not supported."
    )

Personalized model was not supported.


In [29]:
def personalized_recommendations(
    customer_id,
    customer_history,
    item_similarity,
    popularity_df,
    k=10
):
    
    seen_products = set(
        customer_history
    )
    
    candidate_scores = Counter()
    
    for history_product in seen_products:
        
        neighbors = item_similarity.get(
            history_product,
            {}
        )
        
        for candidate_product, similarity_score in (
            neighbors.items()
        ):
            
            if candidate_product not in seen_products:
                
                candidate_scores[
                    candidate_product
                ] += similarity_score
    
    recommendations = []
    
    for product_id, score in (
        candidate_scores
        .most_common()
    ):
        
        recommendations.append({
            "product_id": product_id,
            "score": float(score),
            "method": "personalized_item_cooccurrence"
        })
        
        if len(recommendations) >= k:
            break
    
    if len(recommendations) < k:
        
        already_recommended = set(
            row["product_id"]
            for row in recommendations
        )
        
        fallback_recommendations = (
            popularity_recommendations(
                popularity_df,
                seen_products=(
                    seen_products
                    |
                    already_recommended
                ),
                k=k - len(recommendations)
            )
        )
        
        for row in fallback_recommendations:
            
            recommendations.append({
                "product_id": row["product_id"],
                "score": row["score"],
                "method": "personalized_plus_popularity_fallback"
            })
    
    return recommendations[:k]

In [30]:
def precision_at_k(
    recommendations,
    actual_products,
    k=10
):
    
    recommended_products = [
        row["product_id"]
        for row in recommendations[:k]
    ]
    
    actual_products = set(
        actual_products
    )
    
    if not recommended_products:
        return 0.0
    
    hits = sum(
        product_id in actual_products
        for product_id in recommended_products
    )
    
    return hits / len(recommended_products)


def recall_at_k(
    recommendations,
    actual_products,
    k=10
):
    
    recommended_products = set(
        row["product_id"]
        for row in recommendations[:k]
    )
    
    actual_products = set(
        actual_products
    )
    
    if not actual_products:
        return 0.0
    
    hits = len(
        recommended_products
        &
        actual_products
    )
    
    return hits / len(actual_products)


def hit_rate_at_k(
    recommendations,
    actual_products,
    k=10
):
    
    recommended_products = {
        row["product_id"]
        for row in recommendations[:k]
    }
    
    actual_products = set(
        actual_products
    )
    
    return float(
        len(
            recommended_products
            &
            actual_products
        )
        > 0
    )


def catalog_coverage(
    recommendation_lists,
    total_catalog_size
):
    
    recommended_products = set()
    
    for recommendations in recommendation_lists:
        
        recommended_products.update(
            row["product_id"]
            for row in recommendations
        )
    
    if total_catalog_size == 0:
        return 0.0
    
    return (
        len(recommended_products)
        /
        total_catalog_size
    )

In [31]:
def evaluate_recommendation_method(
    train_data,
    test_data,
    method_name,
    popularity_df,
    item_similarity=None,
    k=10
):
    
    test_customer_products = (
        test_data
        .groupby("customer_id")["product_id"]
        .apply(list)
        .to_dict()
    )
    
    train_customer_products = (
        train_data
        .groupby("customer_id")["product_id"]
        .apply(list)
        .to_dict()
    )
    
    evaluation_rows = []
    recommendation_lists = []
    
    for customer_id, actual_products in (
        test_customer_products.items()
    ):
        
        history = train_customer_products.get(
            customer_id,
            []
        )
        
        if (
            method_name
            ==
            "personalized_item_cooccurrence"
            and
            item_similarity is not None
            and
            len(history) >= MIN_HISTORY_FOR_PERSONALIZATION
        ):
            
            recommendations = (
                personalized_recommendations(
                    customer_id=customer_id,
                    customer_history=history,
                    item_similarity=item_similarity,
                    popularity_df=popularity_df,
                    k=k
                )
            )
        
        else:
            
            recommendations = (
                popularity_recommendations(
                    popularity_df,
                    seen_products=set(history),
                    k=k
                )
            )
        
        recommendation_lists.append(
            recommendations
        )
        
        evaluation_rows.append({
            "customer_id": customer_id,
            "actual_products": actual_products,
            "recommendations": recommendations,
            "precision_at_k": precision_at_k(
                recommendations,
                actual_products,
                k
            ),
            "recall_at_k": recall_at_k(
                recommendations,
                actual_products,
                k
            ),
            "hit_rate_at_k": hit_rate_at_k(
                recommendations,
                actual_products,
                k
            )
        })
    
    detailed_results = pd.DataFrame(
        evaluation_rows
    )
    
    if detailed_results.empty:
        
        return (
            detailed_results,
            pd.DataFrame([{
                "method": method_name,
                "customers_evaluated": 0,
                "precision_at_k": np.nan,
                "recall_at_k": np.nan,
                "hit_rate_at_k": np.nan,
                "catalog_coverage": np.nan
            }])
        )
    
    summary = pd.DataFrame([{
        "method": method_name,
        "customers_evaluated": len(
            detailed_results
        ),
        "precision_at_k": detailed_results[
            "precision_at_k"
        ].mean(),
        "recall_at_k": detailed_results[
            "recall_at_k"
        ].mean(),
        "hit_rate_at_k": detailed_results[
            "hit_rate_at_k"
        ].mean(),
        "catalog_coverage": catalog_coverage(
            recommendation_lists,
            train_data["product_id"].nunique()
        )
    }])
    
    return detailed_results, summary

In [32]:
# Evaluation against the popularity baseline

if not evaluation_supported:

    evaluation_results = pd.DataFrame(
        [
            {
                "method": "popularity_baseline",
                "customers_evaluated": 0,
                "precision_at_k": np.nan,
                "recall_at_k": np.nan,
                "hit_rate_at_k": np.nan,
                "catalog_coverage": np.nan,
                "evaluation_status": (
                    "not_supported"
                ),
                "evaluation_reason": (
                    "No valid customer-level "
                    "holdout interactions were "
                    "available."
                )
            }
        ]
    )

    print(
        "⚠️ Recommendation evaluation was not "
        "statistically supported."
    )

    print(
        "No Precision@K, Recall@K, or Hit Rate@K "
        "will be reported as valid metrics."
    )

else:

    baseline_metrics = evaluate_recommendations(
        test_interactions=test_interactions,
        recommendation_function=(
            popularity_recommendations
        ),
        popularity_df=training_popularity,
        k=TOP_K
    )

    evaluation_results = pd.DataFrame(
        [
            {
                "method": (
                    "popularity_baseline"
                ),
                "customers_evaluated": (
                    baseline_metrics[
                        "customers_evaluated"
                    ]
                ),
                "precision_at_k": (
                    baseline_metrics[
                        "precision_at_k"
                    ]
                ),
                "recall_at_k": (
                    baseline_metrics[
                        "recall_at_k"
                    ]
                ),
                "hit_rate_at_k": (
                    baseline_metrics[
                        "hit_rate_at_k"
                    ]
                ),
                "catalog_coverage": (
                    baseline_metrics[
                        "catalog_coverage"
                    ]
                ),
                "evaluation_status": (
                    "valid"
                ),
                "evaluation_reason": (
                    "Valid temporal holdout "
                    "was available."
                )
            }
        ]
    )


display(
    evaluation_results
)

⚠️ Recommendation evaluation was not statistically supported.
No Precision@K, Recall@K, or Hit Rate@K will be reported as valid metrics.


,method,customers_evaluated,precision_at_k,recall_at_k,hit_rate_at_k,catalog_coverage,evaluation_status,evaluation_reason
0,popularity_baseline,0,NaN,NaN,NaN,NaN,not_supported,No valid customer-level holdout interactions w...


In [33]:
def choose_final_method(
    evaluation_summary_df
):
    
    baseline_row = (
        evaluation_summary_df[
            evaluation_summary_df["method"]
            ==
            "popularity_baseline"
        ]
    )
    
    personalized_rows = (
        evaluation_summary_df[
            evaluation_summary_df["method"]
            ==
            "personalized_item_cooccurrence"
        ]
    )
    
    if personalized_rows.empty:
        return "popularity_baseline"
    
    personalized_row = (
        personalized_rows.iloc[0]
    )
    
    baseline_row = (
        baseline_row.iloc[0]
    )
    
    personalized_hit_rate = (
        personalized_row["hit_rate_at_k"]
    )
    
    baseline_hit_rate = (
        baseline_row["hit_rate_at_k"]
    )
    
    personalized_recall = (
        personalized_row["recall_at_k"]
    )
    
    baseline_recall = (
        baseline_row["recall_at_k"]
    )
    
    if (
        personalized_hit_rate > baseline_hit_rate
        or
        (
            personalized_hit_rate
            ==
            baseline_hit_rate
            and
            personalized_recall
            >
            baseline_recall
        )
    ):
        
        return (
            "personalized_item_cooccurrence"
        )
    
    return "popularity_baseline"


final_primary_method = choose_final_method(
    evaluation_summary_df
)

print(
    f"Selected primary recommendation method: "
    f"{final_primary_method}"
)

Selected primary recommendation method: popularity_baseline


In [36]:
# Fast final recommendation generation
# Popularity baseline is used because personalized
# recommendations are unsupported.

# --------------------------------------------------
# 1. Build customer-product history
# --------------------------------------------------

customer_history_df = (
    interactions_df[
        [
            "customer_id",
            "product_id"
        ]
    ]
    .drop_duplicates()
    .copy()
)


# --------------------------------------------------
# 2. Build final popularity ranking
# --------------------------------------------------

final_popularity = (
    build_popularity_ranking(
        interactions_df
    )
    .copy()
)


if final_popularity.empty:

    raise RuntimeError(
        "Final popularity ranking is empty."
    )


# --------------------------------------------------
# 3. Create candidate recommendations by
#    cross-joining customers with top products
# --------------------------------------------------

customer_ids_df = (
    customer_history_df[
        [
            "customer_id"
        ]
    ]
    .drop_duplicates()
    .copy()
)


top_products_df = (
    final_popularity[
        [
            "product_id",
            "popularity_score"
        ]
    ]
    .head(
        max(
            TOP_K * 3,
            TOP_K
        )
    )
    .copy()
)


# Cross join customers with candidate products

candidate_df = (
    customer_ids_df
    .merge(
        top_products_df,
        how="cross"
    )
)


# --------------------------------------------------
# 4. Remove products already purchased by each
#    customer
# --------------------------------------------------

candidate_df = (
    candidate_df
    .merge(
        customer_history_df.assign(
            already_seen=1
        ),
        on=[
            "customer_id",
            "product_id"
        ],
        how="left"
    )
)


candidate_df = (
    candidate_df[
        candidate_df[
            "already_seen"
        ]
        .isna()
    ]
    .drop(
        columns=[
            "already_seen"
        ]
    )
)


# --------------------------------------------------
# 5. Rank recommendations per customer
# --------------------------------------------------

candidate_df = (
    candidate_df
    .sort_values(
        [
            "customer_id",
            "popularity_score"
        ],
        ascending=[
            True,
            False
        ]
    )
)


candidate_df[
    "rank"
] = (
    candidate_df
    .groupby(
        "customer_id"
    )
    .cumcount()
    + 1
)


candidate_df = (
    candidate_df[
        candidate_df[
            "rank"
        ]
        <= TOP_K
    ]
)


# --------------------------------------------------
# 6. Add recommendation metadata
# --------------------------------------------------

customer_history_size_df = (
    customer_history_df
    .groupby(
        "customer_id"
    )["product_id"]
    .nunique()
    .reset_index(
        name=(
            "customer_history_size"
        )
    )
)


known_customer_recommendations_df = (
    candidate_df
    .merge(
        customer_history_size_df,
        on="customer_id",
        how="left"
    )
)


known_customer_recommendations_df[
    "recommendation_score"
] = (
    known_customer_recommendations_df[
        "popularity_score"
    ]
)


known_customer_recommendations_df[
    "recommendation_method"
] = (
    "popularity_baseline"
)


known_customer_recommendations_df[
    "fallback_type"
] = (
    "personalized_unavailable"
)


known_customer_recommendations_df = (
    known_customer_recommendations_df[
        [
            "customer_id",
            "product_id",
            "rank",
            "recommendation_score",
            "recommendation_method",
            "fallback_type",
            "customer_history_size"
        ]
    ]
    .rename(
        columns={
            "product_id": (
                "recommended_product_id"
            )
        }
    )
)


if (
    known_customer_recommendations_df
    .empty
):

    raise RuntimeError(
        "No known-customer recommendations "
        "were generated."
    )


print(
    "✅ Known-customer recommendations generated."
)


print(
    f"Rows: "
    f"{len(known_customer_recommendations_df):,}"
)


display(
    known_customer_recommendations_df.head(
        20
    )
)

✅ Known-customer recommendations generated.
Rows: 986,660


,customer_id,recommended_product_id,rank,recommendation_score,recommendation_method,fallback_type,customer_history_size
0,00010242fe8c5a6d1ba2dd792cb16214,00018f77f2f0320c557190d7a144bdd3,1,1.0,popularity_baseline,personalized_unavailable,1
1,00010242fe8c5a6d1ba2dd792cb16214,000229ec398224ef6ca0657da4fc703e,2,1.0,popularity_baseline,personalized_unavailable,1
2,00010242fe8c5a6d1ba2dd792cb16214,00024acbcdf0a6daa1e931b038114c75,3,1.0,popularity_baseline,personalized_unavailable,1
3,00010242fe8c5a6d1ba2dd792cb16214,00042b26cf59d7ce69dfabb4e55b4fd9,4,1.0,popularity_baseline,personalized_unavailable,1
4,00010242fe8c5a6d1ba2dd792cb16214,00048cc3ae777c65dbb7d2a0634bc1ea,5,1.0,popularity_baseline,personalized_unavailable,1
5,00010242fe8c5a6d1ba2dd792cb16214,00054e8431b9d7675808bcb819fb4a32,6,1.0,popularity_baseline,personalized_unavailable,1
6,00010242fe8c5a6d1ba2dd792cb16214,000576fe39319847cbb9d288c5617fa6,7,1.0,popularity_baseline,personalized_unavailable,1
7,00010242fe8c5a6d1ba2dd792cb16214,0005a1a1728c9d785b8e2b08b904576c,8,1.0,popularity_baseline,personalized_unavailable,1
8,00010242fe8c5a6d1ba2dd792cb16214,0005f50442cb953dcd1d21e1fb923495,9,1.0,popularity_baseline,personalized_unavailable,1
9,00010242fe8c5a6d1ba2dd792cb16214,00061f2a7bc09da83e415a52dc8a4af1,10,1.0,popularity_baseline,personalized_unavailable,1


In [37]:
cold_start_recommendations = (
    popularity_recommendations(
        final_popularity,
        seen_products=set(),
        k=TOP_K
    )
)

cold_start_rows = []

for rank, recommendation in enumerate(
    cold_start_recommendations,
    start=1
):
    
    cold_start_rows.append({
        "customer_id": None,
        "recommended_product_id": (
            recommendation["product_id"]
        ),
        "rank": rank,
        "recommendation_score": (
            recommendation["score"]
        ),
        "recommendation_method": (
            "popularity_baseline"
        ),
        "fallback_type": (
            "new_customer"
        ),
        "customer_history_size": 0
    })

cold_start_recommendations_df = pd.DataFrame(
    cold_start_rows
)

final_recommendations_df = pd.concat(
    [
        known_customer_recommendations_df,
        cold_start_recommendations_df
    ],
    ignore_index=True
)

print(
    "Cold-start recommendations generated."
)

display(
    cold_start_recommendations_df
)

Cold-start recommendations generated.


,customer_id,recommended_product_id,rank,recommendation_score,recommendation_method,fallback_type,customer_history_size
0,None,00010242fe8c5a6d1ba2dd792cb16214,1,1.0,popularity_baseline,new_customer,0
1,None,00018f77f2f0320c557190d7a144bdd3,2,1.0,popularity_baseline,new_customer,0
2,None,000229ec398224ef6ca0657da4fc703e,3,1.0,popularity_baseline,new_customer,0
3,None,00024acbcdf0a6daa1e931b038114c75,4,1.0,popularity_baseline,new_customer,0
4,None,00042b26cf59d7ce69dfabb4e55b4fd9,5,1.0,popularity_baseline,new_customer,0
5,None,00048cc3ae777c65dbb7d2a0634bc1ea,6,1.0,popularity_baseline,new_customer,0
6,None,00054e8431b9d7675808bcb819fb4a32,7,1.0,popularity_baseline,new_customer,0
7,None,000576fe39319847cbb9d288c5617fa6,8,1.0,popularity_baseline,new_customer,0
8,None,0005a1a1728c9d785b8e2b08b904576c,9,1.0,popularity_baseline,new_customer,0
9,None,0005f50442cb953dcd1d21e1fb923495,10,1.0,popularity_baseline,new_customer,0


In [38]:
recommendation_validation = {
    "total_recommendation_rows": len(
        final_recommendations_df
    ),
    "known_customer_rows": len(
        known_customer_recommendations_df
    ),
    "cold_start_rows": len(
        cold_start_recommendations_df
    ),
    "unique_recommended_products": (
        final_recommendations_df[
            "recommended_product_id"
        ].nunique()
    ),
    "customers_with_recommendations": (
        known_customer_recommendations_df[
            "customer_id"
        ].nunique()
    ),
    "maximum_rank": (
        final_recommendations_df[
            "rank"
        ].max()
    ),
    "null_recommended_products": int(
        final_recommendations_df[
            "recommended_product_id"
        ].isna().sum()
    )
}

recommendation_validation_df = pd.DataFrame(
    [recommendation_validation]
)

display(
    recommendation_validation_df
)

if (
    recommendation_validation[
        "total_recommendation_rows"
    ]
    ==
    0
):
    
    raise RuntimeError(
        "Recommendation generation produced zero rows."
    )

if (
    recommendation_validation[
        "null_recommended_products"
    ]
    >
    0
):
    
    raise RuntimeError(
        "Some recommendation rows contain null products."
    )

if (
    recommendation_validation[
        "unique_recommended_products"
    ]
    ==
    0
):
    
    raise RuntimeError(
        "No unique recommended products were generated."
    )

print(
    "Recommendation validation passed."
)

,total_recommendation_rows,known_customer_rows,cold_start_rows,unique_recommended_products,customers_with_recommendations,maximum_rank,null_recommended_products
0,986670,986660,10,11,98666,10,0


Recommendation validation passed.


In [39]:
recommendation_quality_rows = [
    {
        "validation_name": "raw_interaction_rows",
        "value": interaction_validation[
            "raw_rows"
        ],
        "status": "PASS"
    },
    {
        "validation_name": "missing_customer_ids",
        "value": interaction_validation[
            "missing_customer_ids"
        ],
        "status": (
            "PASS"
            if interaction_validation[
                "missing_customer_ids"
            ] == 0
            else "WARNING"
        )
    },
    {
        "validation_name": "missing_product_ids",
        "value": interaction_validation[
            "missing_product_ids"
        ],
        "status": (
            "PASS"
            if interaction_validation[
                "missing_product_ids"
            ] == 0
            else "WARNING"
        )
    },
    {
        "validation_name": "duplicate_customer_product_rows_after_cleaning",
        "value": int(
            interactions_df.duplicated(
                subset=[
                    "customer_id",
                    "product_id"
                ]
            ).sum()
        ),
        "status": "PASS"
    },
    {
        "validation_name": "recommendation_rows",
        "value": recommendation_validation[
            "total_recommendation_rows"
        ],
        "status": "PASS"
    },
    {
        "validation_name": "unique_recommended_products",
        "value": recommendation_validation[
            "unique_recommended_products"
        ],
        "status": "PASS"
    },
    {
        "validation_name": "evaluation_split_method",
        "value": split_method,
        "status": "PASS"
    },
    {
        "validation_name": "selected_primary_method",
        "value": final_primary_method,
        "status": "PASS"
    }
]

recommendation_quality_df = pd.DataFrame(
    recommendation_quality_rows
)

display(
    recommendation_quality_df
)

,validation_name,value,status
0,raw_interaction_rows,112650,PASS
1,missing_customer_ids,0,PASS
2,missing_product_ids,0,PASS
3,duplicate_customer_product_rows_after_cleaning,0,PASS
4,recommendation_rows,986670,PASS
5,unique_recommended_products,11,PASS
6,evaluation_split_method,no_valid_date,PASS
7,selected_primary_method,popularity_baseline,PASS


In [40]:
powerbi_product_recommendations_df = (
    final_recommendations_df.copy()
)

powerbi_product_recommendations_df[
    "recommendation_generated_at"
] = pd.Timestamp.now()

powerbi_product_recommendations_df[
    "source_interaction_mode"
] = interaction_source["mode"]

powerbi_product_recommendations_df[
    "source_schema"
] = (
    interaction_source.get(
        "interaction_schema"
    )
    or
    interaction_source.get(
        "order_schema"
    )
)

powerbi_product_recommendations_df[
    "source_table"
] = (
    interaction_source.get(
        "interaction_table"
    )
    or
    interaction_source.get(
        "order_table"
    )
)

powerbi_product_recommendations_df[
    "evaluation_split_method"
] = split_method

powerbi_product_recommendations_df[
    "selected_primary_method"
] = final_primary_method

print(
    "Power BI-ready recommendation dataset created."
)

display(
    powerbi_product_recommendations_df.head(20)
)

Power BI-ready recommendation dataset created.


,customer_id,recommended_product_id,rank,recommendation_score,recommendation_method,fallback_type,customer_history_size,recommendation_generated_at,source_interaction_mode,source_schema,source_table,evaluation_split_method,selected_primary_method
0,00010242fe8c5a6d1ba2dd792cb16214,00018f77f2f0320c557190d7a144bdd3,1,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
1,00010242fe8c5a6d1ba2dd792cb16214,000229ec398224ef6ca0657da4fc703e,2,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
2,00010242fe8c5a6d1ba2dd792cb16214,00024acbcdf0a6daa1e931b038114c75,3,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
3,00010242fe8c5a6d1ba2dd792cb16214,00042b26cf59d7ce69dfabb4e55b4fd9,4,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
4,00010242fe8c5a6d1ba2dd792cb16214,00048cc3ae777c65dbb7d2a0634bc1ea,5,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
5,00010242fe8c5a6d1ba2dd792cb16214,00054e8431b9d7675808bcb819fb4a32,6,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
6,00010242fe8c5a6d1ba2dd792cb16214,000576fe39319847cbb9d288c5617fa6,7,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
7,00010242fe8c5a6d1ba2dd792cb16214,0005a1a1728c9d785b8e2b08b904576c,8,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
8,00010242fe8c5a6d1ba2dd792cb16214,0005f50442cb953dcd1d21e1fb923495,9,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline
9,00010242fe8c5a6d1ba2dd792cb16214,00061f2a7bc09da83e415a52dc8a4af1,10,1.0,popularity_baseline,personalized_unavailable,1,2026-07-27 16:54:46.282655,direct,public,order_items,no_valid_date,popularity_baseline


In [41]:
recommendations_csv_path = (
    PREDICTIONS_DIR
    /
    "product_recommendations.csv"
)

evaluation_csv_path = (
    PREDICTIONS_DIR
    /
    "recommendation_evaluation.csv"
)

quality_csv_path = (
    PREDICTIONS_DIR
    /
    "recommendation_data_quality.csv"
)

powerbi_csv_path = (
    PREDICTIONS_DIR
    /
    "powerbi_product_recommendations.csv"
)

final_recommendations_df.to_csv(
    recommendations_csv_path,
    index=False
)

evaluation_summary_df.to_csv(
    evaluation_csv_path,
    index=False
)

recommendation_quality_df.to_csv(
    quality_csv_path,
    index=False
)

powerbi_product_recommendations_df.to_csv(
    powerbi_csv_path,
    index=False
)

print("CSV outputs saved successfully.")

print(recommendations_csv_path)
print(evaluation_csv_path)
print(quality_csv_path)
print(powerbi_csv_path)

print("\nFiles currently in data\\predictions:")
for file_path in sorted(
    PREDICTIONS_DIR.iterdir()
):
    if file_path.is_file():
        print(file_path.name)

CSV outputs saved successfully.
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\product_recommendations.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\recommendation_evaluation.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\recommendation_data_quality.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\powerbi_product_recommendations.csv

Files currently in data\predictions:
customer_churn_model_comparison.csv
customer_churn_prediction_distribution.csv
customer_churn_predictions.csv
customer_churn_test_metrics.csv
powerbi_product_recommendations.csv
product_recommendations.csv
recommendation_data_quality.csv
recommendation_evaluation.csv


In [42]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

encoded_password = quote_plus(DB_PASSWORD)

sqlalchemy_url = (
    f"postgresql+psycopg2://"
    f"{quote_plus(DB_USER)}:"
    f"{encoded_password}@"
    f"{DB_HOST}:"
    f"{DB_PORT}/"
    f"{DB_NAME}"
)

engine = create_engine(
    sqlalchemy_url,
    pool_pre_ping=True,
    future=True
)

recommendation_schema = "recommendations"

with engine.begin() as sqlalchemy_connection:
    
    sqlalchemy_connection.execute(
        text(
            f"""
            CREATE SCHEMA IF NOT EXISTS
            "{recommendation_schema}";
            """
        )
    )

print(
    f"PostgreSQL output schema is ready: "
    f"{recommendation_schema}"
)

PostgreSQL output schema is ready: recommendations


In [44]:
# Write recommendation outputs to PostgreSQL safely

recommendations_table_name = (
    "product_recommendations"
)

evaluation_table_name = (
    "recommendation_evaluation"
)

quality_table_name = (
    "recommendation_data_quality"
)

powerbi_table_name = (
    "powerbi_product_recommendations"
)


# --------------------------------------------------
# Validate PostgreSQL engine
# --------------------------------------------------

if engine is None:

    raise RuntimeError(
        "SQLAlchemy engine is not available."
    )


# --------------------------------------------------
# Ensure target schema exists
# --------------------------------------------------

with engine.begin() as connection:

    connection.execute(
        text(
            f"""
            CREATE SCHEMA IF NOT EXISTS
            "{recommendation_schema}"
            """
        )
    )


# --------------------------------------------------
# Helper function for safe DataFrame loading
# --------------------------------------------------

def write_dataframe_to_postgresql(
    dataframe,
    table_name,
    engine,
    schema,
    chunksize=5000
):

    if dataframe is None:

        raise ValueError(
            f"{table_name}: DataFrame is None."
        )


    if dataframe.empty:

        print(
            f"⚠️ {table_name} is empty."
        )

        return


    print(
        f"Writing {table_name}: "
        f"{len(dataframe):,} rows..."
    )


    dataframe.to_sql(
        name=table_name,
        con=engine,
        schema=schema,
        if_exists="replace",
        index=False,
        chunksize=chunksize,
        method=None
    )


    print(
        f"✅ {table_name} written successfully."
    )


# --------------------------------------------------
# Write final recommendation outputs
# --------------------------------------------------

write_dataframe_to_postgresql(
    dataframe=final_recommendations_df,
    table_name=recommendations_table_name,
    engine=engine,
    schema=recommendation_schema
)


write_dataframe_to_postgresql(
    dataframe=evaluation_summary_df,
    table_name=evaluation_table_name,
    engine=engine,
    schema=recommendation_schema
)


write_dataframe_to_postgresql(
    dataframe=recommendation_quality_df,
    table_name=quality_table_name,
    engine=engine,
    schema=recommendation_schema
)


write_dataframe_to_postgresql(
    dataframe=powerbi_product_recommendations_df,
    table_name=powerbi_table_name,
    engine=engine,
    schema=recommendation_schema
)


print(
    "✅ Recommendation outputs written to PostgreSQL."
)


print(
    f"{recommendation_schema}."
    f"{recommendations_table_name}"
)


print(
    f"{recommendation_schema}."
    f"{evaluation_table_name}"
)


print(
    f"{recommendation_schema}."
    f"{quality_table_name}"
)


print(
    f"{recommendation_schema}."
    f"{powerbi_table_name}"
)

Writing product_recommendations: 986,670 rows...
✅ product_recommendations written successfully.
Writing recommendation_evaluation: 1 rows...
✅ recommendation_evaluation written successfully.
Writing recommendation_data_quality: 8 rows...
✅ recommendation_data_quality written successfully.
Writing powerbi_product_recommendations: 986,670 rows...
✅ powerbi_product_recommendations written successfully.
✅ Recommendation outputs written to PostgreSQL.
recommendations.product_recommendations
recommendations.recommendation_evaluation
recommendations.recommendation_data_quality
recommendations.powerbi_product_recommendations


In [45]:
verification_connection = None

try:
    
    verification_connection = get_psycopg2_connection()
    
    verification_query = """
        SELECT
            table_schema,
            table_name
        FROM information_schema.tables
        WHERE table_schema = %s
          AND table_name IN (%s, %s, %s, %s)
        ORDER BY table_name;
    """
    
    with verification_connection.cursor() as cursor:
        
        cursor.execute(
            verification_query,
            (
                recommendation_schema,
                recommendations_table_name,
                evaluation_table_name,
                quality_table_name,
                powerbi_table_name
            )
        )
        
        verified_tables = cursor.fetchall()
    
    verification_connection.commit()
    
    verified_tables_df = pd.DataFrame(
        verified_tables,
        columns=[
            "table_schema",
            "table_name"
        ]
    )
    
    print(
        "Verified PostgreSQL recommendation tables:"
    )
    
    display(
        verified_tables_df
    )
    
    expected_tables = {
        recommendations_table_name,
        evaluation_table_name,
        quality_table_name,
        powerbi_table_name
    }
    
    actual_tables = set(
        verified_tables_df["table_name"]
    )
    
    missing_tables = (
        expected_tables
        -
        actual_tables
    )
    
    if missing_tables:
        raise RuntimeError(
            f"PostgreSQL verification failed. "
            f"Missing tables: {missing_tables}"
        )
    
    print(
        "PostgreSQL table verification passed."
    )

except Exception as error:
    
    safe_rollback(
        verification_connection
    )
    
    raise error

finally:
    
    close_connection(
        verification_connection
    )

Verified PostgreSQL recommendation tables:


,table_schema,table_name
0,recommendations,powerbi_product_recommendations
1,recommendations,product_recommendations
2,recommendations,recommendation_data_quality
3,recommendations,recommendation_evaluation


PostgreSQL table verification passed.


In [46]:
verification_connection = None

try:
    
    verification_connection = get_psycopg2_connection()
    
    output_columns_query = """
        SELECT
            table_schema,
            table_name,
            ordinal_position,
            column_name,
            data_type,
            udt_name,
            is_nullable
        FROM information_schema.columns
        WHERE table_schema = %s
          AND table_name IN (%s, %s, %s, %s)
        ORDER BY
            table_name,
            ordinal_position;
    """
    
    with verification_connection.cursor() as cursor:
        
        cursor.execute(
            output_columns_query,
            (
                recommendation_schema,
                recommendations_table_name,
                evaluation_table_name,
                quality_table_name,
                powerbi_table_name
            )
        )
        
        output_columns = cursor.fetchall()
        
        output_column_names = [
            description[0]
            for description in cursor.description
        ]
    
    verification_connection.commit()
    
    output_columns_df = pd.DataFrame(
        output_columns,
        columns=output_column_names
    )
    
    print(
        "Verified PostgreSQL output column structures:"
    )
    
    display(
        output_columns_df
    )

except Exception as error:
    
    safe_rollback(
        verification_connection
    )
    
    raise error

finally:
    
    close_connection(
        verification_connection
    )

Verified PostgreSQL output column structures:


,table_schema,table_name,ordinal_position,column_name,data_type,udt_name,is_nullable
0,recommendations,powerbi_product_recommendations,1,customer_id,text,text,YES
1,recommendations,powerbi_product_recommendations,2,recommended_product_id,text,text,YES
2,recommendations,powerbi_product_recommendations,3,rank,bigint,int8,YES
3,recommendations,powerbi_product_recommendations,4,recommendation_score,double precision,float8,YES
4,recommendations,powerbi_product_recommendations,5,recommendation_method,text,text,YES
5,recommendations,powerbi_product_recommendations,6,fallback_type,text,text,YES
6,recommendations,powerbi_product_recommendations,7,customer_history_size,bigint,int8,YES
7,recommendations,powerbi_product_recommendations,8,recommendation_generated_at,timestamp without time zone,timestamp,YES
8,recommendations,powerbi_product_recommendations,9,source_interaction_mode,text,text,YES
9,recommendations,powerbi_product_recommendations,10,source_schema,text,text,YES


In [47]:
powerbi_validation = {
    "recommendation_table_exists": (
        (
            verified_tables_df["table_name"]
            ==
            recommendations_table_name
        ).any()
    ),
    "powerbi_table_exists": (
        (
            verified_tables_df["table_name"]
            ==
            powerbi_table_name
        ).any()
    ),
    "recommendation_rows": len(
        final_recommendations_df
    ),
    "powerbi_rows": len(
        powerbi_product_recommendations_df
    ),
    "unique_customers": (
        known_customer_recommendations_df[
            "customer_id"
        ].nunique()
    ),
    "unique_recommended_products": (
        final_recommendations_df[
            "recommended_product_id"
        ].nunique()
    ),
    "maximum_rank": (
        final_recommendations_df[
            "rank"
        ].max()
    ),
    "recommendation_score_available": (
        final_recommendations_df[
            "recommendation_score"
        ].notna().any()
    ),
    "recommendation_method_available": (
        final_recommendations_df[
            "recommendation_method"
        ].notna().all()
    ),
    "fallback_type_available": (
        final_recommendations_df[
            "fallback_type"
        ].notna().all()
    )
}

powerbi_validation_df = pd.DataFrame(
    [
        {
            "validation": key,
            "result": value
        }
        for key, value in powerbi_validation.items()
    ]
)

display(
    powerbi_validation_df
)

required_validations = [
    "recommendation_table_exists",
    "powerbi_table_exists",
    "recommendation_score_available",
    "recommendation_method_available",
    "fallback_type_available"
]

for validation_name in required_validations:
    
    if not powerbi_validation[
        validation_name
    ]:
        
        raise RuntimeError(
            f"Power BI validation failed: "
            f"{validation_name}"
        )

print(
    "Power BI-ready validation passed."
)

,validation,result
0,recommendation_table_exists,True
1,powerbi_table_exists,True
2,recommendation_rows,986670
3,powerbi_rows,986670
4,unique_customers,98666
5,unique_recommended_products,11
6,maximum_rank,10
7,recommendation_score_available,True
8,recommendation_method_available,True
9,fallback_type_available,True


Power BI-ready validation passed.


In [48]:
print("=" * 80)
print("NOTEBOOK 12 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nRecommendation method:")
print(final_primary_method)

print("\nEvaluation method:")
print(split_method)

print("\nInteraction source:")
print(interaction_source["mode"])

print("\nFinal recommendation rows:")
print(
    len(
        final_recommendations_df
    )
)

print("\nUnique recommended products:")
print(
    final_recommendations_df[
        "recommended_product_id"
    ].nunique()
)

print("\nCSV outputs:")
print(
    recommendations_csv_path
)
print(
    evaluation_csv_path
)
print(
    quality_csv_path
)
print(
    powerbi_csv_path
)

print("\nPostgreSQL outputs:")
print(
    f"{recommendation_schema}."
    f"{recommendations_table_name}"
)
print(
    f"{recommendation_schema}."
    f"{evaluation_table_name}"
)
print(
    f"{recommendation_schema}."
    f"{quality_table_name}"
)
print(
    f"{recommendation_schema}."
    f"{powerbi_table_name}"
)

print("\nModel artifact:")
print(
    "No serialized model artifact required."
)

print("\nNotebook 12 is complete.")

NOTEBOOK 12 COMPLETED SUCCESSFULLY

Recommendation method:
popularity_baseline

Evaluation method:
no_valid_date

Interaction source:
direct

Final recommendation rows:
986670

Unique recommended products:
11

CSV outputs:
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\product_recommendations.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\recommendation_evaluation.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\recommendation_data_quality.csv
C:\Users\asus\Documents\ECOMMERCE-AI-PLATFORM\data\predictions\powerbi_product_recommendations.csv

PostgreSQL outputs:
recommendations.product_recommendations
recommendations.recommendation_evaluation
recommendations.recommendation_data_quality
recommendations.powerbi_product_recommendations

Model artifact:
No serialized model artifact required.

Notebook 12 is complete.
